# 04 — Error Analysis

Where and when the final model fails, per ML1 doc sections 35-38.
Requires `python -m src.train --stage final` to have been run.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 60)


In [ ]:
from src import config, error_analysis, explain

reports = error_analysis.run()
list(reports)

## Headline test metrics

In [ ]:
pd.DataFrame({h: r['overall'] for h, r in reports.items()}).T

## Bias

A large mean signed error means the model is systematically high or low, not just noisy.

In [ ]:
pd.Series({h: r['bias_mean_error'] for h, r in reports.items()}, name='mean signed error')

## Per-mandi performance

Overall accuracy hides mandis where the model is much worse.

In [ ]:
for horizon, report in reports.items():
    print(f'\n=== +{horizon} ===')
    display(pd.DataFrame(report['by_mandi']))

## Per-variety and per-grade

In [ ]:
for horizon, report in reports.items():
    print(f'\n=== +{horizon} ===')
    display(pd.DataFrame(report['by_variety']))
    display(pd.DataFrame(report['by_grade']))

## Regimes

Rapid price moves and unusual arrival volumes are where a forecast is both hardest and most valuable.

In [ ]:
for horizon, report in reports.items():
    print(f'\n=== +{horizon} ===')
    display(pd.DataFrame(report['by_price_regime']))
    display(pd.DataFrame(report['by_arrivals_regime']))
    display(pd.DataFrame(report['by_season']))

## Worst individual predictions

In [ ]:
pd.DataFrame(reports['1d']['worst_25_predictions'])

## Feature importance

Permutation importance answers the question that matters: how much does
accuracy actually depend on this column? It is association, not causation.

In [ ]:
importance_path = config.IMPORTANCE_DIR / 'explanation_hist_gbr.json'
if importance_path.exists():
    import json
    data = json.loads(importance_path.read_text(encoding='utf-8'))
    for horizon in ('1d', '3d', '7d'):
        top = data[horizon]['permutation_importance']['mae_increase_when_shuffled']
        display(pd.Series(top, name=f'+{horizon} MAE increase').head(15))
else:
    print('run: python -m src.explain --model hist_gbr')